# w9_i2ce_t.ipynb — τ 扫描,5 折 × {512, 4096}

User (2026-07-19): τ=0.02 从战役开局冻结至今从未调过——它是剂量旋钮的
第三根轴(单步 CE 锐度),tag 案的未审共犯。臂:**i2cet05**(τ=0.05)/
**i2cet10**(τ=0.10)/**i2ce_icetl**(可学 τ,init 0.02,log 参数化、
无 wd、inv_t 夹 [5,200],ckpt 时打印当前 τ)。τ=0.02 参照 = 第一波
ce/i2ce 五折(read-only 引入)。3 臂 × 5 折 × {512, 2048} = 30 塔,2000ep,
seed=fold 配对,ZS-only,rvsel。**4096 半区已搁置(用户:成本太高),
先在 512(~5.6G)/2048(~22G)看效果**——两 cap 都在 L40 顶棚内,
便宜 pod 也能整场跑。τ=0.02 参照 = 第一波 ce/i2ce 的 512/2048 折。判决问题:软 τ 是
沿 non-tag 前沿滑动(τ 无罪,剂量论再证),还是存在 τ* 同时 non≥.70
tag≥.73(τ = 比 swin 更便宜的剂量阀)?可学 τ 顺带回答"模型想要多尖"。
AUTO-STOPS。


In [ ]:
# constants
import os

REPO = os.path.abspath("..")   # this release folder (contains Pod/ and VICReg_review/)
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_cv_out"       # CV campaign out dir

RECIPES = ["wcle_i2cet05_icetf",   # tau = 0.05
           "wcle_i2cet10_icetf",   # tau = 0.10
           "wcle_i2ce_icetl"]      # learnable tau (init 0.02)
REF_RECIPES = ["wcle_ce_cetf", "wcle_i2ce_icetf"]   # tau=0.02 wave-1 rows
CAPS = [512, 2048]   # 4096 shelved (user: cost too high)
# CAPS = [512, 4096]
N_FOLDS = 5
EPOCHS, CKPT_EVERY, CKPT_SEEDS, TOPUP_SEEDS = 2000, 50, 2, 10   # ZS: seeds unused

# VRAM scheduler knobs
SAFETY = 0.85
RESERVE_GIB = 1.5

# 48G cards (L40) can't fit the 4096 grad-gallery cells (~45G): pods whose
# smallest GPU has <60GiB free skip caps above MAX_CAP_48G, run their share
# (512/1024/2048 = 30 towers) and AUTO-STOP early -- no waiting on the
# A100 pod, no OOM-burned claims. 80G pods run all 40.
MAX_CAP_48G = 2048

def nm_of(r, k, cap):
    return f"w9cv_{r}_fold{k}" + (f"_g{cap}" if cap != 512 else "")

os.makedirs(OUT_DIR, exist_ok=True)
print("jobs:", len(RECIPES) * N_FOLDS * len(CAPS),
      f"({len(RECIPES)} recipes x {N_FOLDS} folds x {len(CAPS)} caps) @ {EPOCHS}ep")


In [ ]:
# Local setup (release build: the code ships with this folder -- no
# repository synchronisation is needed or performed).
import importlib.util
import os
import sys
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        %pip -q install scikit-learn scipy
        break
os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")

In [ ]:
# Stage the corpus into RAM (same file set as w9_a100.ipynb).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz", "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)

In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# VRAM-BUDGET SCHEDULER over the full (recipe x fold x cap) grid.
# Warmup measures one cost per cap (i2ce fold0, the heavier recipe); the
# ZS-only done marker is tower_<nm>_fp_ep{EPOCHS}.npz (a lower-budget or
# crashed tower auto-continues from its newest ckpt / resume bundle).
import os, subprocess, tempfile, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
# measure runs write to a LOCAL scratch dir: with the real job name on the
# SHARED volume, a measure could load another machine's resume bundle
# (start_ep >= 1 -> zero steps -> no cost file) or race its zs_traj writes.
MEAS_OUT = os.path.join(tempfile.gettempdir(), "w9_measure_out")
os.makedirs(MEAS_OUT, exist_ok=True)
gpus = J.detect_gpus()

def _smi_mib(field, g):
    out = subprocess.check_output(
        ["nvidia-smi", f"--query-gpu={field}", "--format=csv,noheader,nounits",
         "-i", str(g)]).decode().strip().split("\n")[0]
    return int(out) * 2**20

free = {g: _smi_mib("memory.free", g) for g in gpus}
budget = {g: int(free[g] * SAFETY - RESERVE_GIB * 2**30) for g in gpus}
print(f"[vram] budgets {[f'{budget[g] / 2**30:.0f}G' for g in gpus]}")
vram_gib = min(free.values()) / 2**30
CAP_CEIL = MAX_CAP_48G if vram_gib < 60 else 10**9
print(f"[vram] cap ceiling " + (str(CAP_CEIL) if CAP_CEIL < 10**9
                                else "none (runs all caps)"))

todo0 = []
for cap in CAPS:
    for r in RECIPES:
        for k in range(N_FOLDS):
            nm = nm_of(r, k, cap)
            if cap > CAP_CEIL:
                print(f"[skip-vram] {nm} cap {cap} > {CAP_CEIL}"); continue
            if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS}.npz").exists():
                print(f"[skip] {nm} at {EPOCHS}"); continue
            todo0.append((r, k, cap, nm))

cost = {}
for cap in sorted({c for _r, _k, c, _n in todo0}):
    # the three tau arms are architecturally IDENTICAL i2ce (tau is a
    # scalar): one cost per CAP, measured on the first recipe.
    tf = Path(tempfile.gettempdir()) / f"w9cv_vram_g{cap}.txt"
    tf.unlink(missing_ok=True)
    cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
           MEAS_OUT, "--repo", REPO, "--arm", RECIPES[0], "--fold", "0",
           "--n-folds", str(N_FOLDS), "--anchor-cap", str(cap),
           "--epochs", "1", "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--measure-vram", str(tf)]
    print(f"[warmup] cap {cap} ...", flush=True)
    with open(logd / f"measure_cv_g{cap}.log", "w") as fh:
        subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                       env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpus[0]))
    cost[cap] = int(tf.read_text()) if tf.exists() else budget[gpus[0]] + 1
    print(f"[warmup] cap {cap}: {cost[cap] / 2**30:.2f}G", flush=True)

todo = sorted(((r, k, cap, nm, cost[cap]) for r, k, cap, nm in todo0),
              key=lambda x: -x[4])            # 4096 half sets the makespan
now_used = {g: 0 for g in gpus}
fails = []
cvn = threading.Condition()

def run_job(g, r, k, cap, nm, c):
    try:
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
        cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
               OUT_DIR, "--repo", REPO, "--arm", r, "--fold", str(k),
               "--n-folds", str(N_FOLDS), "--anchor-cap", str(cap),
               "--epochs", str(EPOCHS), "--ckpt-every", str(CKPT_EVERY),
               "--ckpt-seeds", str(CKPT_SEEDS), "--topup-seeds", str(TOPUP_SEEDS),
               "--full-pool", "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        t0 = time.time()
        with open(logd / f"{r}_fold{k}_g{cap}.log", "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=g))
        if p.returncode != 0:
            (cdir / f"{nm}.claim").unlink(missing_ok=True); fails.append(nm)
        print(f"[gpu{g}] {'ok' if p.returncode == 0 else 'FAIL'} {nm} "
              f"[{(time.time() - t0) / 3600:.1f}h]", flush=True)
    finally:
        with cvn:
            now_used[g] -= c
            cvn.notify_all()

stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
active = []
with cvn:
    pending = list(todo)
    while pending or any(t.is_alive() for t in active):
        prog = False
        i = 0
        while i < len(pending):
            r, k, cap, nm, c = pending[i]
            fit = [g for g in gpus if now_used[g] + c <= budget[g] or now_used[g] == 0]
            if not fit:
                i += 1; continue
            g = min(fit, key=lambda g: now_used[g])
            now_used[g] += c
            th = threading.Thread(target=run_job, args=(g, r, k, cap, nm, c),
                                  daemon=True)
            active.append(th); th.start(); pending.pop(i)
            print(f"[sched] {nm} -> gpu{g} ({c / 2**30:.1f}G, used "
                  f"{now_used[g] / 2**30:.1f}/{budget[g] / 2**30:.0f}G)", flush=True)
            prog = True
        active = [t for t in active if t.is_alive()]
        if not prog:
            cvn.wait(timeout=3)
stop_evt.set()
for t in active:
    t.join()
print(f"FINAL grid drained; {len(fails)} failed")
for nm in fails:
    print("  FAILED:", nm)


In [ ]:
# SELECTION-REPAIR (rerun-safe): towers finished under the OLD selection
# (zsbest without 'rvsel') are re-selected from their saved projections --
# the worker skips training (done-marker), refreshes the traj from the npzs
# (SPq stored per ckpt -> no GPU re-embedding) and rewrites zsbest by rvsel.
import json as _json, os, queue as _q, subprocess, threading, time
from pathlib import Path

need = []
for cap in CAPS:
    for r in RECIPES:
        for k in range(N_FOLDS):
            nm = nm_of(r, k, cap)
            if not (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS}.npz").exists():
                continue
            zb = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
            if zb.exists() and "rvsel" in _json.loads(zb.read_text()):
                continue
            need.append((r, k, cap, nm))
print(f"{len(need)} tower(s) need re-selection")
jobs = _q.Queue()
for j in need:
    jobs.put(j)
cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"

def rep(g):
    while True:
        try:
            r, k, cap, nm = jobs.get_nowait()
        except _q.Empty:
            return
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held -- skipped", flush=True); continue
        cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
               OUT_DIR, "--repo", REPO, "--arm", r, "--fold", str(k),
               "--n-folds", str(N_FOLDS), "--anchor-cap", str(cap),
               "--epochs", str(EPOCHS), "--full-pool",
               "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        t0 = time.time()
        with open(logd / f"reselect_{nm}.log", "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=g))
        print(f"[gpu{g}] reselect {'ok' if p.returncode == 0 else 'FAIL'} {nm} "
              f"[{(time.time() - t0) / 60:.0f}m]", flush=True)

ths = [threading.Thread(target=rep, args=(g,)) for g in J.detect_gpus()]
for t in ths:
    t.start()
for t in ths:
    t.join()
print("re-selection pass done")


In [ ]:
# THE PAPER TABLE: per (recipe, cap) 5-fold mean+-std + PAIRED per-fold
# delta (i2ce - ce shares split AND seed within each fold).
import json
import numpy as np
from pathlib import Path

def _load(nm):
    p = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    return json.loads(p.read_text()) if p.exists() else None

for cap in CAPS:
    print(f"===== cap {cap} =====")
    per = {}
    for r in RECIPES + REF_RECIPES:
        rows = [(_load(nm_of(r, k, cap)), k) for k in range(N_FOLDS)]
        rows = [(d, k) for d, k in rows if d]
        per[r] = {k: d for d, k in rows}
        lab = r.replace("wcle_", "").replace("_icetf", "").replace("_cetf", "")
        if not rows:
            print(f"  {lab:6s} 0/{N_FOLDS} (none)"); continue
        def ms(key):
            v = [d[key] for d, _k in rows]; return np.mean(v), np.std(v)
        def msg(key):
            v = [d.get(key) for d, _k in rows if d.get(key) is not None]
            return (np.mean(v), np.std(v)) if v else (float("nan"), float("nan"))
        n1, n5, tg, nu = ms("nm_noname"), ms("h5_noname"), ms("tag_noname"), ms("nm_neutral")
        q1, q5, qt = msg("t_q1"), msg("t_q5"), msg("t_qtag")
        print(f"  {lab:6s} {len(rows)}/{N_FOLDS}  non@1 {n1[0]:.3f}+-{n1[1]:.3f}  "
              f"non@5 {n5[0]:.3f}+-{n5[1]:.3f}  tag_non {tg[0]:.3f}+-{tg[1]:.3f}  "
              f"neu {nu[0]:.3f}+-{nu[1]:.3f}")
        print(f"         REVIEW(test): q@1 {q1[0]:.3f}+-{q1[1]:.3f}  "
              f"q@5 {q5[0]:.3f}+-{q5[1]:.3f}  qtag {qt[0]:.3f}+-{qt[1]:.3f}"
              "   (nan = pre-rvsel zsbest; run the repair cell)")
    a, b = "wcle_ce_cetf", "wcle_i2ce_icetf"
    both = sorted(set(per.get(a, {})) & set(per.get(b, {})))
    if both:
        for key, klab in (("nm_noname", "non@1"), ("tag_noname", "tag_non")):
            ds = [per[b][k][key] - per[a][k][key] for k in both]
            wins = sum(d > 0 for d in ds)
            print(f"  PAIRED d(i2ce-ce) {klab}: "
                  + " ".join(f"{d:+.3f}" for d in ds)
                  + f"  | mean {np.mean(ds):+.3f}  wins {wins}/{len(ds)}")
    print()


In [ ]:
# AUTO-STOP removed in the release build: stopping the machine is cloud-
# provider tooling, not part of the experiment. All results are already on
# the shared volume when the run cells finish.
print("run complete -- results are in", OUT_DIR)